In [16]:
import pymongo as py
import pandas as pd
import pandas as pd
import pymongo as py
import mysql.connector as sql
import plotly.express as px
from streamlit_option_menu import option_menu
import plotly.graph_objects as go

client=py.MongoClient("mongodb://localhost:27017")
db=client['AirBNB']

database= sql.connect(host="localhost",user ="root",password ="kobalan",auth_plugin="mysql_native_password",database="airbnb")
Cursor=database.cursor()


In [23]:
#Room_DATA
data_LIST1=[]
for i in db.Data.find( {}, {'_id':1,'listing_url':1,'name':1,'property_type':1,'room_type':1,'bed_type':1,
                        'minimum_nights':1,'maximum_nights':1,'cancellation_policy':1,'accommodates':1,
                        'bedrooms':1,'beds':1,'number_of_reviews':1,'bathrooms':1,'price':1,
                        'cleaning_fee':1,'extra_people':1,'guests_included':1,'images.picture_url':1,
                        'review_scores.review_scores_rating':1} ):
        data_LIST1.append(i)


df_ROOMS=pd.DataFrame(data_LIST1)
df_ROOMS['review_scores'] = df_ROOMS['review_scores'].apply(lambda x: x.get('review_scores_rating',0))
df_ROOMS.fillna({'bedrooms':0},inplace=True)
df_ROOMS.fillna({'beds':0},inplace=True)
df_ROOMS.fillna({'bathrooms':0},inplace=True)
df_ROOMS.fillna({'cleaning_fee':0},inplace=True)

#Type_CONVERSION
df_ROOMS['minimum_nights']=df_ROOMS['minimum_nights'].astype(int)
df_ROOMS['maximum_nights']=df_ROOMS['maximum_nights'].astype(int)
df_ROOMS['bedrooms']=df_ROOMS['bedrooms'].astype(int)
df_ROOMS['beds']=df_ROOMS['beds'].astype(int)
df_ROOMS['bathrooms']=df_ROOMS['bathrooms'].astype(int)
df_ROOMS['price']=df_ROOMS['price'].astype(int)
df_ROOMS['cleaning_fee']=df_ROOMS['cleaning_fee'].apply(lambda x: int(x) if x!='Not Specified' else 'Not Specified')

df_ROOMS[df_ROOMS['_id']=='10059244']


,_id,listing_url,name,property_type,room_type,bed_type,minimum_nights,maximum_nights,cancellation_policy,accommodates,bedrooms,beds,number_of_reviews,bathrooms,price,cleaning_fee,extra_people,guests_included,images,review_scores
10,10059244,https://www.airbnb.com/rooms/10059244,Ligne verte - à 15 min de métro du centre ville.,Apartment,Entire home/apt,Real Bed,2,1125,moderate,2,0,1,0,1,43,0,12,1,{'picture_url': 'https://a0.muscache.com/im/pi...,0


In [25]:

data_LIST2=[]
for i in db.Data.find({},{'id':1,'host':1}):
        data_LIST2.append(i)

#Removing Unneccessary Data
df_HOST= pd.DataFrame(data_LIST2)
host_keys = list(df_HOST.iloc[0,1].keys())
host_keys.remove('host_about')

for i in host_keys:
    if i == 'host_response_time':
        df_HOST['host_response_time'] = df_HOST['host'].apply(lambda x: x['host_response_time'] if 'host_response_time' in x else 'Not Specified')
    else:
        df_HOST[i] = df_HOST['host'].apply(lambda x: x[i] if i in x and x[i]!='' else 'Not Specified')

df_HOST.drop(columns=['host'], inplace=True)

df_HOST[df_HOST['_id']=='10038496']
# df_HOST

,_id,host_id,host_url,host_name,host_location,host_response_time,host_thumbnail_url,host_picture_url,host_neighbourhood,host_response_rate,host_is_superhost,host_has_profile_pic,host_identity_verified,host_listings_count,host_total_listings_count,host_verifications
9,10038496,51530266,https://www.airbnb.com/users/show/51530266,Ana Valéria,"Rio de Janeiro, State of Rio de Janeiro, Brazil",within an hour,https://a0.muscache.com/im/pictures/8c7bb5fe-7...,https://a0.muscache.com/im/pictures/8c7bb5fe-7...,Copacabana,100,True,True,True,2,2,"[email, phone, reviews, jumio, government_id]"


In [28]:

#Address
data_LIST3=[]
for i in db.Data.find({},{'id':1,'address':1}):
    data_LIST3.append(i)

df_ADDRESS=pd.DataFrame(data_LIST3)
address_KEYS=list(df_ADDRESS.iloc[0,1].keys())


# address_KEYS

#Now we are going to split the address row into different row

for i in address_KEYS:
    if i == 'location':
        df_ADDRESS['location_type'] = df_ADDRESS['address'].apply(lambda x: x['location']['type'])
        df_ADDRESS['longitude'] = df_ADDRESS['address'].apply(lambda x: x['location']['coordinates'][0])
        df_ADDRESS['latitude'] = df_ADDRESS['address'].apply(lambda x: x['location']['coordinates'][1])
        #df_ADDRESS['is_location_exact'] = df_ADDRESS['address'].apply(lambda x: x['location']['is_location_exact'])
    else:
        df_ADDRESS[i] = df_ADDRESS['address'].apply(lambda x: x[i] if x[i]!='' else 'Not Specified')

df_ADDRESS.drop(columns=['address','street'], inplace=True)
#df_ADDRESS['is_location_exact'] = df_ADDRESS['is_location_exact'].map({False:'No',True:'Yes'})

df_ADDRESS[df_ADDRESS['_id']=='10038496']
df_ADDRESS['country'].unique()


array(['United States', 'Turkey', 'Hong Kong', 'Australia', 'Portugal',
       'Brazil', 'Canada', 'Spain', 'China'], dtype=object)

In [26]:

availability = []
for i in db.Data.find( {}, {'_id':1, 'availability':1}):
    availability.append(i)

df_availability = pd.DataFrame(availability)
availability_keys = list(df_availability.iloc[0,1].keys())


for i in availability_keys:
    df_availability['availability_30'] = df_availability['availability'].apply(lambda x: x['availability_30'])
    df_availability['availability_60'] = df_availability['availability'].apply(lambda x: x['availability_60'])
    df_availability['availability_90'] = df_availability['availability'].apply(lambda x: x['availability_90'])
    df_availability['availability_365'] = df_availability['availability'].apply(lambda x: x['availability_365'])

df_availability.drop(columns=['availability'], inplace=True)
df_availability.head()

,_id,availability_30,availability_60,availability_90,availability_365
0,1003530,0,0,0,93
1,10133554,30,60,90,365
2,10059872,0,0,0,0
3,10084023,14,24,40,220
4,10091713,0,0,0,0


In [29]:

df = pd.merge(df_ROOMS, df_HOST, on='_id')
df = pd.merge(df, df_ADDRESS, on='_id')
df = pd.merge(df, df_availability, on='_id')
df.to_csv('AirBNB_Analysis.csv', index=False)


In [2]:
#df.to_csv('AirBNB_Analaysis3.csv',index=False,encoding="utf-8")
import pandas as pd
import plotly.figure_factory as ff
df=pd.read_csv('AirBNB_Analaysis.csv')
#df_1 = df[["name","property_type","room_type","bed_type","minimum_nights","maximum_nights","price","cancellation_policy","accommodates","bedrooms","beds","number_of_reviews","bathrooms","extra_people","guests_included","review_scores"]]
# fig = ff.create_table(df_1.iloc[0:5])

# fig.show()

# df_1['price']

In [14]:
#"name","property_type","room_type","bed_type","minimum_nights",maximum_nights","cancellation_policy","accommodates	bedrooms","beds","number_of_reviews","bathrooms	price","cleaning_fee","extra_people	guests_included","review_scores"
# df_sample.head(2)
df_1=df[["country","name","property_type","room_type","bed_type","minimum_nights","maximum_nights","cancellation_policy","accommodates","bedrooms","beds","number_of_reviews","bathrooms","price","extra_people","guests_included","review_scores"]]
#df_1.sort_values(by='country')
# df_1.reset_index()
#df_1.iloc[:500, 1:20:2].style.background_gradient(cmap="Oranges")
df_1['country'].unique().tolist()

['United States',
 'Turkey',
 'Hong Kong',
 'Australia',
 'Portugal',
 'Brazil',
 'Canada',
 'Spain',
 'China']

In [13]:
df_1.head(2)

,country,name,property_type,room_type,bed_type,minimum_nights,maximum_nights,cancellation_policy,accommodates,bedrooms,beds,number_of_reviews,bathrooms,price,extra_people,guests_included,review_scores
0,United States,New York City - Upper West Side Apt,Apartment,Private room,Real Bed,12,360,strict_14_with_grace_period,2,1,1,70,1,1,0,1,94
1,Turkey,Double and triple rooms Blue mosque,Bed and breakfast,Private room,Real Bed,1,1125,moderate,3,1,2,29,1,1,0,1,92


In [37]:
df_1=df[["country","name","property_type","room_type","bed_type","cancellation_policy","minimum_nights","maximum_nights","accommodates","bedrooms","beds","number_of_reviews","bathrooms","price","extra_people","guests_included","review_scores"]]
#df_1.sort_values(by='price',ascending=False,ignore_index=True)
df2 = df_1.reset_index(drop=True)
df2.head()
# fig = ff.create_table(df_1.iloc[0:5])

df_3=df[["host_id","host_name","host_location","host_response_time","host_picture_url","host_neighbourhood","host_response_rate","host_is_superhost","host_has_profile_pic","host_identity_verified","host_listings_count","	host_total_listings_count"]]



,country,name,property_type,room_type,bed_type,cancellation_policy,minimum_nights,maximum_nights,accommodates,bedrooms,beds,number_of_reviews,bathrooms,price,extra_people,guests_included,review_scores
0,United States,New York City - Upper West Side Apt,Apartment,Private room,Real Bed,strict_14_with_grace_period,12,360,2,1,1,70,1,135,0,1,94
1,Turkey,Double and triple rooms Blue mosque,Bed and breakfast,Private room,Real Bed,moderate,1,1125,3,1,2,29,1,121,0,1,92
2,Hong Kong,"Soho Cozy, Spacious and Convenient",Apartment,Entire home/apt,Real Bed,flexible,4,20,3,1,2,3,1,699,0,1,100
3,Hong Kong,City center private room with bed,Guesthouse,Private room,Futon,strict_14_with_grace_period,1,500,1,1,1,81,1,181,100,1,92
4,Australia,Surry Hills Studio - Your Perfect Base in Sydney,Apartment,Entire home/apt,Real Bed,strict_14_with_grace_period,10,21,2,0,1,64,1,181,0,1,95


In [ ]:
"host_id","host_name","host_location","host_response_time","host_picture_url","host_neighbourhood","host_response_rate","host_is_superhost","host_has_profile_pic","host_identity_verified","host_listings_count","	host_total_listings_count"


In [6]:
from PIL import Image
import os
import matplotlib.pyplot as plt

Folder_Path='E:/Python/Phonepe/pulse/Img'
Folder_Files=[os.path.join(Folder_Path,File) for File in os.listdir(Folder_Path) if File.endswith(('jpg'))]
print(Folder_Files)

width=[]
height=[]
modes=[]

for i in range(0,len(Folder_Files)):
    image=Image.open(Folder_Files[i])
    width.append(image.size[0])
    height.append(image.size[1])
    modes.append(image.mode)

    
for i in range(0,len(Folder_Files)):
    img=Image.open(Folder_Files[i])
    print(img)
    # plt.imshow(image)
    # plt.show()

print(width,height,modes)

['E:/Python/Phonepe/pulse/Img\\home_img2.jpg', 'E:/Python/Phonepe/pulse/Img\\home_img3.jpg']
<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=2880x822 at 0x200B336A7B0>
<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=3840x1096 at 0x200B336ACF0>
[2880, 3840] [822, 1096] ['RGB', 'RGB']
